# First test dataset -- minimal, well-posed, few parameters

Before testing parameter-estimation methods on the harder cases (nonlinear, chaotic,
underdetermined, structural error, ...), we want one dataset where the answer is simply
*checkable*: a unique `x_true` exists and can be recovered exactly. That rules out most of
`data_generator_funs`:

- **`generate_linear_dataset`** is the only generator where `Y = X @ B` is solvable in closed
  form (least squares) -- no ODE integration, no chaos, no emulator needed to check the answer.
- **`noise_std=0.0`, `structural_error_level=0.0`**: "has a solution" specifically means no
  observation noise and no structural bias baked into `y_true`, so an exact `x_true` exists that
  reproduces `y_true` -- not just an approximate best fit.
- **`structure="full"`**: dense coefficients, no sparsity pattern to worry about yet.
- **`n_outputs=10 >= n_params=5`**: more outputs than parameters, so the 5 parameters are
  identifiable (avoids the underdetermined case) -- both kept small for a fast sanity check.

This is the simplest point in the challenge spectrum from `CLAUDE.md`: everything harder
(noise, structural error, sparsity, nonlinearity, chaos, conflicting models, too-few/too-many
targets) is a deliberate step away from this baseline.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import numpy as np

from data_generator_funs.dataset import generate_linear_dataset
from utils.plotting import plot_correlation, plot_data_overview

In [2]:
import pandas as pd

In [5]:
ds = generate_linear_dataset(
    n_samples=200, n_params=10, n_outputs=30, structure="full",
    noise_std=0.0, structural_idx=[], structural_error_level=0.0,
    seed=3,
    nc_path="/glade/u/home/qingyuany/repos/cliffnotes/datasets",
)

{k: v.shape for k, v in ds.items() if hasattr(v, "shape")}

{'X': (200, 10),
 'Y': (200, 30),
 'x_true': (10,),
 'y_true': (30,),
 'paras_to_vary': (0,),
 'outputs_to_vary': (0,),
 'structural_bias': (30,)}

In [6]:
import xarray as xr

xx = xr.open_dataset('/glade/u/home/qingyuany/repos/cliffnotes/datasets/linear_full_n200_p10_o30/linear_full_n200_p10_o30.nc')
X = pd.read_csv('/glade/u/home/qingyuany/repos/cliffnotes/datasets/linear_full_n200_p10_o30/X.csv', index_col= 0)
Y = pd.read_csv('/glade/u/home/qingyuany/repos/cliffnotes/datasets/linear_full_n200_p10_o30/Y.csv', index_col= 0)

x_true = pd.read_csv('/glade/work/qingyuany/cliffnotes/linear_full_n200_p10_o30/x_true.csv', index_col= 0)
obs = pd.read_csv('/glade/work/qingyuany/cliffnotes/linear_full_n200_p10_o30/y_true.csv', index_col= 0)



In [20]:
(x_true.iloc[:,0].values @ xx.coefficients.values - obs.iloc[:,0].values)

array([ 4.44089210e-16, -1.73472348e-16,  0.00000000e+00, -3.33066907e-16,
        8.32667268e-17,  0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
        0.00000000e+00,  0.00000000e+00,  1.11022302e-16,  3.33066907e-16,
       -1.11022302e-16,  0.00000000e+00,  4.44089210e-16, -2.22044605e-16,
        6.66133815e-16, -8.32667268e-17,  0.00000000e+00, -2.22044605e-16,
       -4.44089210e-16, -8.32667268e-17,  2.22044605e-16,  2.22044605e-16,
        8.88178420e-16,  0.00000000e+00,  4.44089210e-16,  0.00000000e+00,
       -4.44089210e-16,  4.44089210e-16])

In [18]:
x_true

,0
x0,-0.073052
x1,-0.680114
x2,-0.512836
x3,0.144202
x4,0.238785
x5,0.844959
x6,0.255971
x7,0.126359
x8,0.962992
x9,0.955183


## Check it actually has a solution

Two checks:
1. The training ensemble is exactly noise-free: `X @ B` should match `Y` to floating-point precision.
2. `x_true` should be exactly recoverable from `y_true` by solving `B.T @ x = y_true` (least
   squares, since there are more outputs than parameters) -- this is the "a solution exists"
   property this dataset is for.

In [52]:
import pandas as pd


In [61]:
answer = pd.read_csv('/glade/work/qingyuany/repo_data/linear_test_n100_p30_o100/output/sample1_all_para_realscale.csv', index_col = 0)

In [67]:
answer.min()[13]

/glade/derecho/scratch/qingyuany/tmp/ipykernel_71709/3379289929.py:1: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  answer.min()[13]


-0.3916059621063765

In [66]:
answer.max()[13]

/glade/derecho/scratch/qingyuany/tmp/ipykernel_71709/1317098369.py:1: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  answer.max()[13]


0.9020645934276732

In [65]:
xx['x_true'].values[13]

0.3405251828619642